# Basic

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import polars as pl
import pickle
import numpy as np
import tqdm

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer


In [ ]:
with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)

with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()


In [ ]:

lambdas_list = np.round(np.arange(0, 1.2, 0.2), 1)
lambdas_list_fine = np.round(np.arange(0.81, 1, 0.01), 2)

lambdas_list_fine


# selections by lambda

In [ ]:
# for lam in tqdm.tqdm(lambdas_list):
for lam in tqdm.tqdm(lambdas_list_fine):
    selector = tokenizer.LazyGreedyTokenSelector(combined_subgraphs, mapped_ids, D = config.TokenizerParam().max_dist_candidate, lam = lam)
    history = selector.select(k = len(mapped_ids),n_jobs=-1)   # [(token, marginal_gain, cumulative_score), ...]
    df_history =(
        pl.DataFrame(
        history, schema=["token", "gain", "cumulative_score"], orient="row")
        .with_columns(
        pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
        .with_row_index())
    df_history.write_parquet(f"{config.CandidateLists().path_greedy_tree}{lam}.parquet")
